# MoodTune — Phase 2: Data Cleaning

Creates a track-level derived dataset without altering `data/raw/dataset.csv`. Cleaning decisions are documented in `docs/data_cleaning.md`.

In [ ]:
from pathlib import Path
import pandas as pd

project_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((root for root in project_roots if (root / 'data' / 'raw').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Run this notebook from the MoodTune project or a child directory.')
RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'dataset.csv'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'spotify_cleaned.csv'
raw = pd.read_csv(RAW_PATH)
print(f'Raw shape: {raw.shape}')

In [ ]:
required_columns = {'Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre'}
missing_columns = required_columns.difference(raw.columns)
if missing_columns:
    raise ValueError(f'Unexpected dataset schema; missing columns: {sorted(missing_columns)}')

essential_metadata = ['track_id', 'artists', 'album_name', 'track_name']
clean = raw.drop(columns='Unnamed: 0').dropna(subset=essential_metadata).copy()
clean = clean[clean['track_id'].str.strip().ne('')].copy()
stable_columns = [column for column in clean.columns if column not in {'track_id', 'popularity', 'track_genre'}]
conflicting_columns = [column for column in stable_columns if clean.groupby('track_id')[column].nunique(dropna=False).gt(1).any()]
if conflicting_columns:
    raise ValueError(f'Cannot consolidate inconsistent track records: {conflicting_columns}')

clean['_source_order'] = clean.index
clean = clean.sort_values(['track_id', 'popularity', '_source_order'], ascending=[True, False, True])
aggregation = {column: 'first' for column in stable_columns}
aggregation['popularity'] = 'max'
aggregation['track_genre'] = lambda values: '; '.join(sorted(values.dropna().unique()))
clean = clean.groupby('track_id', as_index=False, sort=True).agg(aggregation).rename(columns={'track_genre': 'track_genres'})

In [ ]:
bounded_features = ['danceability', 'energy', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence']
if clean[essential_metadata].isna().any().any() or clean['track_id'].duplicated().any():
    raise ValueError('Cleaned dataset failed required metadata or uniqueness validation.')
if (clean['duration_ms'] <= 0).any() or any(((clean[column] < 0) | (clean[column] > 1)).any() for column in bounded_features):
    raise ValueError('Cleaned dataset contains invalid retained feature values.')
clean.to_csv(OUTPUT_PATH, index=False)
print(f'Cleaned shape: {clean.shape}')
print(f'Rows removed: {len(raw) - len(clean):,}')
print(f'Output: {OUTPUT_PATH}')
display(clean.head())